In [2]:
from nepPhoneticEncoder import nepali_dmetaphone


print(nepali_dmetaphone("gharja")[0])

GRJA


In [5]:
import json
from nepPhoneticEncoder import nepali_dmetaphone
from gensim.models import FastText

JSON_PATH = "nabin_saksham_merged_with_ids.json"
MODEL_PATH = "FastText/Fast_text_model_for_merged"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    phonetic_data = json.load(f)

code_to_base = {
    code: entry["base_word"]
    for code, entry in phonetic_data.items()
}

base_words_set = {entry["base_word"] for entry in phonetic_data.values()}

model = FastText.load(MODEL_PATH)


In [26]:
def phonetic_candidate(word: str):
    """Return base_word if phonetic code matches, else None."""
    codes = nepali_dmetaphone(word)
    if not codes:
        return None
    code = codes[0]
    # print(code)
    return code_to_base.get(code)  # returns base_word or None


def fasttext_candidate(word: str):
    """
    If the word is in FastText vocab, search top-3 similar words.
    If any of them is a base_word from JSON, return that base_word.
    Else return None.
    """
    try:
        similar = model.wv.most_similar(word, topn=3)
    except KeyError:
        return None  # word not in FastText vocab

    for sim_word, score in similar:
        if sim_word in base_words_set:
            return sim_word
    return None


def normalize_word(word: str) -> str:
    """
    1) Try phonetic match → base_word
    2) If no phonetic match → try FastText and see if any top-3 similar
       words is one of the base_words in JSON.
    3) If neither → return original word.
    """
    # 1) Phonetic first
    base_from_phonetic = phonetic_candidate(word)
    if base_from_phonetic is not None:
        return base_from_phonetic

    # 2) Fall back to FastText
    base_from_fasttext = fasttext_candidate(word)
    if base_from_fasttext is not None:
        return base_from_fasttext

    # 3) Nothing found, keep as-is
    return word


def normalize_sentence(sentence: str) -> str:
    """
    Very simple tokenizer: split on whitespace.
    You can improve this later for punctuation handling.
    """
    words = sentence.split()
    normalized = [normalize_word(w) for w in words]
    return " ".join(normalized)


sentence = "deshko netalai hyerda ksm vakhar gara ghati katdim hunx mug hamro desh vrasta netalai fasi dera marnu parx ani desh x dhotile npli ko power"
print("Original:  ", sentence)
print("Normalized:", normalize_sentence(sentence))


Original:   deshko netalai hyerda ksm vakhar gara ghati katdim hunx mug hamro desh vrasta netalai fasi dera marnu parx ani desh x dhotile npli ko power
Normalized: deshko netalai hyerda ksm vakhar gara ghati katdim hunx megh hamro desh vrasta netalai fasi dera marnu parx ani desh x dhotile npli ko pawar
Normalized: deshko netalai hyerda ksm vakhar gara ghati katdim hunx megh hamro desh vrasta netalai fasi dera marnu parx ani desh x dhotile npli ko pawar


In [27]:
# Normalize the first column of `final_merged.csv` using the pipeline
!pip install pandas
import pandas as pd

INPUT_CSV = 'final_merged.csv'
OUTPUT_CSV = 'normalized.csv'

try:
    df = pd.read_csv(INPUT_CSV, encoding='utf-8', dtype=str)
except FileNotFoundError:
    print(f'File not found: {INPUT_CSV}')
else:
    # pick the first column (by position). Keep its name for output.
    col0 = df.columns[0]
    # ensure no NaNs and apply normalization pipeline defined above
    df[col0] = df[col0].fillna('').apply(normalize_sentence)
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    print(f'Saved {len(df)} rows to {OUTPUT_CSV} (normalized column: {col0})')


Saved 21156 rows to normalized.csv (normalized column: text)
Saved 21156 rows to normalized.csv (normalized column: text)
